# 02 — Features: Numeric + Categorical + Skills Pipeline

**Project H17 — Fair AI Hiring with Bias Audit.** We build the design matrix exactly as the production model uses it (`fair_hiring.features.build_feature_pipeline`) and inspect its dimensionality, scale, and what each leg contributes.

In [ ]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')
sys.path.insert(0, '../src')
from fair_hiring.features import (build_feature_pipeline, skill_columns,
                                  NUMERIC_FEATURES, CATEGORICAL_FEATURES)
df = pd.read_parquet('../data/processed/resumes.parquet')
len(df)

## 1. Build the column-transform pipeline

In [ ]:
pipe = build_feature_pipeline(df)
X = pipe.fit_transform(df)
print(f'design matrix shape: {X.shape}')
print(f'NUMERIC_FEATURES = {NUMERIC_FEATURES}')
print(f'CATEGORICAL_FEATURES = {CATEGORICAL_FEATURES}')
print(f'skill columns: {len(skill_columns(df))}')

## 2. Per-feature mean / std after scaling

In [ ]:
ct = pipe.named_steps['pre']
num_names = NUMERIC_FEATURES + skill_columns(df)
X_num = X[:, :len(num_names)]
stats = pd.DataFrame({'feature': num_names,
                       'mean': X_num.mean(axis=0).round(3),
                       'std': X_num.std(axis=0).round(3)})
print(stats.head(8)); print('...')
print(stats.tail(8))

## 3. Visualise pre vs post-scaling for `years_experience`

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
sns.histplot(df['years_experience'], bins=25, color='#1f77b4', ax=axes[0])
axes[0].set_title('raw years_experience')
sns.histplot(X_num[:, NUMERIC_FEATURES.index('years_experience')], bins=25, color='#2ca02c', ax=axes[1])
axes[1].set_title('scaled years_experience')
plt.tight_layout(); plt.show()

## 4. Education one-hot — block view

In [ ]:
X_cat = X[:, len(num_names):]
print(f'one-hot block shape: {X_cat.shape}')
print(f'sample first 4 candidates:'); print(X_cat[:4])
# Map columns to labels
ohe = ct.named_transformers_['cat']
print(f'edu category names: {ohe.get_feature_names_out(CATEGORICAL_FEATURES).tolist()}')

## 5. Skill-feature density — per-row activation

In [ ]:
skill_block = X_num[:, len(NUMERIC_FEATURES):]
row_norms = np.linalg.norm(skill_block, axis=1)
fig, ax = plt.subplots(figsize=(8, 4))
sns.histplot(row_norms, bins=40, color='#9467bd', ax=ax)
ax.set_title('Per-candidate scaled-skill row L2 norm')
plt.tight_layout(); plt.show()

## 6. Skill density per gender — sanity

In [ ]:
rn_df = pd.DataFrame({'rn': row_norms, 'gender': df['gender'].values})
fig, ax = plt.subplots(figsize=(7, 4))
sns.boxplot(data=rn_df, x='gender', y='rn', palette=['#1f77b4', '#fb6a4a'], ax=ax)
ax.set_title('Skill row-norm by gender — should overlap'); plt.tight_layout(); plt.show()

## 7. Save the engineered design matrix for downstream notebooks

In [ ]:
out = Path('../data/processed/design_matrix.parquet')
X_df = pd.DataFrame(X, columns=num_names + list(ohe.get_feature_names_out(CATEGORICAL_FEATURES)))
X_df['gender'] = df['gender'].values
X_df['nationality_group'] = df['nationality_group'].values
X_df['hire_label'] = df['hire_label'].values
X_df.to_parquet(out, index=False)
print(f'wrote {X_df.shape} -> {out}')

## 8. Implications
- Numeric features (years_experience, prior_employer_tier) and skill block are scaled; education is one-hot; gender / nationality_group are kept as separate columns *outside* the model so they can be used as the sensitive attribute for the audit.
- Skill-row L2 distributions overlap by gender — no obvious systematic skill-feature gap. Any disparate impact must come through `prior_employer_tier` or a correlated education channel.